# 🤖 Tuần 7: Time-Series Transformer cho Nhận diện Cử chỉ Tay Động
## Phase 5 — Self-Attention trên Dữ liệu Chuỗi Tọa độ 3D

---

### 📋 Mục tiêu
- Xây dựng kiến trúc **Transformer Encoder** để phân loại chuỗi cử chỉ tay 15 frame.
- Hiểu cơ chế **Self-Attention** và vai trò của **Positional Encoding** cho dữ liệu chuỗi thời gian.
- Trực quan hóa **Attention Weights** để giải thích cách mô hình "nhìn" dữ liệu.
- So sánh hiệu năng giữa **Transformer** và **LSTM** từ Tuần 6.

---

### 🧠 Tại sao dùng Transformer cho Time-Series?

| Đặc điểm | LSTM | Transformer |
|---|---|---|
| Xử lý chuỗi | Tuần tự (sequential) | Song song (parallel) |
| Phụ thuộc xa | Khó nắm bắt (vanishing gradient) | Dễ dàng qua Self-Attention |
| Tốc độ huấn luyện | Chậm hơn (không song song hóa) | Nhanh hơn (parallelizable) |
| Khả năng giải thích | Hidden states khó giải thích | Attention map trực quan |

Với chuỗi ngắn (15 frames), cả hai kiến trúc đều có thể hoạt động tốt. Tuy nhiên, Transformer có lợi thế ở khả năng **nắm bắt mối quan hệ giữa mọi cặp frame** chỉ trong một bước tính toán — thay vì phải "nhớ" thông tin qua từng bước thời gian như LSTM.

## 1. Import Thư viện & Cấu hình

In [1]:
import os
import math
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from copy import deepcopy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

warnings.filterwarnings('ignore')

# ── Cấu hình đồ họa ──
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'font.family': 'DejaVu Sans',
    'axes.grid': True,
    'grid.alpha': 0.3
})

# ── Bảng màu chuyên nghiệp ──
COLORS = {
    'train_loss': '#2196F3',
    'val_loss': '#F44336',
    'train_acc': '#1976D2',
    'val_acc': '#D32F2F',
    'primary': '#1565C0',
    'secondary': '#E53935',
    'accent': '#43A047',
    'highlight': '#FF8F00',
    'transformer': '#7B1FA2',
    'lstm': '#00897B'
}

# ── Thiết bị tính toán ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# ── Đường dẫn dự án ──
BASE_DIR = r'D:\WORKSAPCE\Monitor_AR\1_AI_Model_Training'
DATA_PATH = os.path.join(BASE_DIR, 'Dataset', 'raw_data.csv')
MODEL_DIR = os.path.join(BASE_DIR, 'Models')
BEST_MODEL_PATH = os.path.join(MODEL_DIR, 'best_transformer.pth')
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Seed toàn cục ──
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"📂 Data: {DATA_PATH}")
print(f"💾 Model save: {BEST_MODEL_PATH}")

ModuleNotFoundError: No module named 'seaborn'

## 2. Nạp và Tiền xử lý Dữ liệu

Mỗi mẫu (sample) là chuỗi **15 frame** liên tiếp, mỗi frame gồm **63 tọa độ** (21 landmarks × 3 trục X/Y/Z).  
Tensor đầu vào cho mô hình có shape: `(Batch, 15, 63)`.

In [ ]:
# ── 2.1 Đọc CSV ──
df = pd.read_csv(DATA_PATH)
print(f"📊 Kích thước DataFrame: {df.shape}")
print(f"📌 Các cột: {list(df.columns[:10])} ... (tổng {len(df.columns)} cột)")
print(f"\n🏷️  Các nhãn cử chỉ ({df['label'].nunique()} lớp):")
print(df['label'].value_counts().to_string())
print(f"\n📈 Số sample: {df['sample_id'].nunique()}")
print(f"📐 Frames/sample: {df.groupby('sample_id')['frame_index'].count().unique()}")
df.head(3)

In [ ]:
# ── 2.2 Xác định cột đặc trưng ──
meta_cols = ['sample_id', 'frame_index', 'label']
feature_cols = [c for c in df.columns if c not in meta_cols]
print(f"🔢 Số cột đặc trưng: {len(feature_cols)}")
assert len(feature_cols) == 63, f"Kỳ vọng 63 cột đặc trưng, nhận được {len(feature_cols)}"

# ── 2.3 LabelEncoder ──
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])
num_classes = len(le.classes_)
print(f"\n🏷️  Ánh xạ nhãn:")
for i, name in enumerate(le.classes_):
    count = (df['label_encoded'] == i).sum() // 15  # mỗi sample có 15 rows
    print(f"   {i}: {name} ({count} samples)")

In [ ]:
# ── 2.4 Tạo ma trận 3D: (num_samples, 15, 63) ──
grouped = df.sort_values(['sample_id', 'frame_index']).groupby('sample_id')

X_list = []
y_list = []

for sample_id, group in grouped:
    features = group[feature_cols].values  # (15, 63)
    label = group['label_encoded'].iloc[0]
    X_list.append(features)
    y_list.append(label)

X_all = np.array(X_list, dtype=np.float32)  # (N, 15, 63)
y_all = np.array(y_list, dtype=np.int64)     # (N,)

print(f"✅ X shape: {X_all.shape}")
print(f"✅ y shape: {y_all.shape}")
print(f"   Phân bố nhãn: {dict(zip(*np.unique(y_all, return_counts=True)))}")

In [ ]:
# ── 2.5 Chia tập: 70/15/15 ──
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_all, y_all, test_size=0.15, random_state=SEED, stratify=y_all
)

# 70/15 từ phần còn lại (85%) → val = 15/85 ≈ 0.1765
val_ratio = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_ratio, random_state=SEED, stratify=y_train_val
)

print(f"📊 Phân chia dữ liệu:")
print(f"   Train : {X_train.shape[0]} samples ({X_train.shape[0]/len(X_all)*100:.1f}%)")
print(f"   Val   : {X_val.shape[0]} samples ({X_val.shape[0]/len(X_all)*100:.1f}%)")
print(f"   Test  : {X_test.shape[0]} samples ({X_test.shape[0]/len(X_all)*100:.1f}%)")

In [ ]:
# ── 2.6 StandardScaler: fit trên Train, transform tất cả ──
num_samples_train, seq_len, num_features = X_train.shape

# Reshape → 2D để fit scaler
X_train_2d = X_train.reshape(-1, num_features)
scaler = StandardScaler()
scaler.fit(X_train_2d)

# Transform từng tập
X_train = scaler.transform(X_train.reshape(-1, num_features)).reshape(X_train.shape)
X_val   = scaler.transform(X_val.reshape(-1, num_features)).reshape(X_val.shape)
X_test  = scaler.transform(X_test.reshape(-1, num_features)).reshape(X_test.shape)

print(f"✅ Chuẩn hóa hoàn tất.")
print(f"   Train mean: {X_train.mean():.6f}, std: {X_train.std():.6f}")
print(f"   Val   mean: {X_val.mean():.6f}, std: {X_val.std():.6f}")
print(f"   Test  mean: {X_test.mean():.6f}, std: {X_test.std():.6f}")

In [ ]:
# ── 2.7 Custom Dataset & DataLoader ──
class GestureSequenceDataset(Dataset):
    """Dataset cho chuỗi cử chỉ tay (15 frame × 63 features)."""
    
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]  # (15, 63), scalar


BATCH_SIZE = 32

train_dataset = GestureSequenceDataset(X_train, y_train)
val_dataset   = GestureSequenceDataset(X_val, y_val)
test_dataset  = GestureSequenceDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# Kiểm tra 1 batch
sample_x, sample_y = next(iter(train_loader))
print(f"✅ Batch shape: X={sample_x.shape}, y={sample_y.shape}")
print(f"   Kỳ vọng: X=(batch, 15, 63), y=(batch,)")

---
## 3. Kiến trúc Transformer

### 🔑 Self-Attention: Cốt lõi của Transformer

Self-Attention cho phép mỗi frame "nhìn" tất cả các frame khác trong chuỗi và tính trọng số quan trọng. Công thức:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Trong đó:
- **Q** (Query), **K** (Key), **V** (Value) được tạo từ embedding của mỗi frame.
- $\frac{1}{\sqrt{d_k}}$ là hệ số chuẩn hóa, ngăn gradient vanishing khi $d_k$ lớn.
- Ma trận attention có kích thước **(15 × 15)**: mỗi ô cho biết frame $i$ chú ý bao nhiêu đến frame $j$.

### ⏱️ Tại sao cần Positional Encoding?

Self-Attention **không có khái niệm thứ tự** — nó xem input như tập hợp (set), không phải chuỗi (sequence). Để mô hình biết "frame 0 đến trước frame 14", ta thêm **Positional Encoding** bằng hàm sin/cos:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Mỗi vị trí được mã hóa thành vector duy nhất, giúp mô hình phân biệt thứ tự thời gian.

In [ ]:
# ── 3.1 Positional Encoding ──
class PositionalEncoding(nn.Module):
    """
    Mã hóa vị trí bằng hàm sin/cos theo công thức chuẩn.
    Input:  (batch, seq_len, d_model)
    Output: (batch, seq_len, d_model)  — cùng shape, đã cộng thêm vị trí.
    """
    
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Tạo ma trận PE: (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)  # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )  # (d_model/2,)
        
        pe[:, 0::2] = torch.sin(position * div_term)  # Vị trí chẵn: sin
        pe[:, 1::2] = torch.cos(position * div_term)  # Vị trí lẻ:  cos
        
        # Thêm batch dimension: (1, max_len, d_model)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)  # Không cập nhật gradient
    
    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# ── Minh họa Positional Encoding ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pe_demo = PositionalEncoding(d_model=128, max_len=15, dropout=0.0)
pe_values = pe_demo.pe.squeeze(0).numpy()  # (15, 128)

# Heatmap
im = axes[0].imshow(pe_values, cmap='RdBu_r', aspect='auto', interpolation='nearest')
axes[0].set_xlabel('Chiều d_model')
axes[0].set_ylabel('Vị trí (Frame)')
axes[0].set_title('Positional Encoding Heatmap (15 frames × 128 dims)')
plt.colorbar(im, ax=axes[0], fraction=0.046)

# Waveforms cho một số chiều
for dim_idx in [0, 1, 4, 5, 20, 21]:
    axes[1].plot(range(15), pe_values[:, dim_idx], label=f'dim {dim_idx}', linewidth=1.5)
axes[1].set_xlabel('Vị trí (Frame)')
axes[1].set_ylabel('Giá trị PE')
axes[1].set_title('PE Waveforms cho các chiều khác nhau')
axes[1].legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()
print("💡 Mỗi vị trí có pattern sin/cos duy nhất — giúp Transformer phân biệt thứ tự frame.")

In [ ]:
# ── 3.2 Gesture Transformer Model ──
class GestureTransformer(nn.Module):
    """
    Transformer Encoder cho phân loại chuỗi cử chỉ tay.
    
    Pipeline:
        Input (batch, 15, 63)
          → Linear Projection (63 → 128)
          → Positional Encoding
          → Transformer Encoder (3 layers, 4 heads)
          → Global Mean Pooling (trung bình 15 frames)
          → Classifier Head (128 → 64 → num_classes)
    """
    
    def __init__(self, input_dim=63, d_model=128, nhead=4,
                 dim_feedforward=256, num_layers=3, num_classes=10,
                 dropout=0.2, pe_dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        
        # ① Input Projection: 63 → 128
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # ② Positional Encoding
        self.pos_encoder = PositionalEncoding(
            d_model=d_model, max_len=500, dropout=pe_dropout
        )
        
        # ③ Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,         # Input: (batch, seq, feature)
            activation='relu'
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )
        
        # ④ Classifier Head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        
        # Lưu attention weights để trực quan hóa
        self._attention_weights = None
        self._hooks = []
    
    def forward(self, x):
        # x: (batch, 15, 63)
        
        # ① Projection
        x = self.input_projection(x)  # (batch, 15, 128)
        
        # ② Positional Encoding
        x = self.pos_encoder(x)  # (batch, 15, 128)
        
        # ③ Transformer Encoder
        x = self.transformer_encoder(x)  # (batch, 15, 128)
        
        # ④ Global Mean Pooling — trung bình qua chiều thời gian
        x = x.mean(dim=1)  # (batch, 128)
        
        # ⑤ Classification
        x = self.classifier(x)  # (batch, num_classes)
        
        return x
    
    def get_attention_maps(self, x):
        """
        Forward pass với hook để thu thập attention weights.
        Trả về: (output, list_of_attention_weights)
        Mỗi attention weight có shape: (batch, nhead, seq_len, seq_len)
        """
        attention_maps = []
        hooks = []
        
        # Đăng ký hook cho từng layer
        for layer in self.transformer_encoder.layers:
            def hook_fn(module, input_args, output, attn_list=attention_maps):
                # Gọi self_attn riêng để lấy attention weights
                pass  # Sẽ dùng cách khác bên dưới
        
        # Cách trực tiếp hơn: chạy thủ công qua từng layer
        self.eval()
        with torch.no_grad():
            # Projection + PE
            out = self.input_projection(x)
            out = self.pos_encoder(out)
            
            # Từng Transformer layer
            for layer in self.transformer_encoder.layers:
                # MultiheadAttention forward với need_weights=True
                # Layer norm trước (pre-norm) hoặc sau (post-norm) tùy cấu hình
                # Mặc định PyTorch dùng post-norm
                src2, attn_weights = layer.self_attn(
                    out, out, out, need_weights=True,
                    average_attn_weights=False
                )
                attention_maps.append(attn_weights)  # (batch, nhead, seq, seq)
                
                # Hoàn thành forward pass của layer
                src2 = layer.dropout1(src2)
                out = layer.norm1(out + src2)
                src2 = layer.linear2(layer.dropout(layer.activation(layer.linear1(out))))
                src2 = layer.dropout2(src2)
                out = layer.norm2(out + src2)
            
            # Pooling + Classifier
            pooled = out.mean(dim=1)
            output = self.classifier(pooled)
        
        return output, attention_maps


# ── Khởi tạo mô hình ──
model = GestureTransformer(
    input_dim=63,
    d_model=128,
    nhead=4,
    dim_feedforward=256,
    num_layers=3,
    num_classes=num_classes,
    dropout=0.2,
    pe_dropout=0.1
).to(device)

print(model)
print(f"\n{'='*50}")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"📊 Tổng tham số: {total_params:,}")
print(f"🏋️  Tham số huấn luyện: {trainable_params:,}")

### 🏗️ Kiến trúc Transformer — Sơ đồ luồng dữ liệu

```
Input: (batch, 15, 63)
         │
         ▼
┌─────────────────────┐
│  Linear Projection  │  63 → 128
│    (Input Embed)    │
└────────┬────────────┘
         │
         ▼
┌─────────────────────┐
│ Positional Encoding │  + sin/cos vectors
│   + Dropout(0.1)    │
└────────┬────────────┘
         │
    ╔════╧════════════════════╗
    ║  Transformer Encoder    ║
    ║  ┌───────────────────┐  ║
    ║  │ Multi-Head Attn   │  ║  × 3 layers
    ║  │ (4 heads, d=128)  │  ║
    ║  ├───────────────────┤  ║
    ║  │ Feed-Forward Net  │  ║  dim_ff=256
    ║  │ + LayerNorm       │  ║
    ║  └───────────────────┘  ║
    ╚════╤════════════════════╝
         │
         ▼  (batch, 15, 128)
┌─────────────────────┐
│ Global Mean Pooling │  mean over 15 frames
└────────┬────────────┘
         │  (batch, 128)
         ▼
┌─────────────────────┐
│  Linear(128→64)     │
│  ReLU + Drop(0.3)   │
│  Linear(64→classes) │
└─────────────────────┘
         │
         ▼
   Output: (batch, num_classes)
```

---
## 4. Huấn luyện Mô hình

**Chiến lược huấn luyện:**
- **Optimizer:** Adam (lr=5e-4, weight_decay=1e-4)
- **Loss:** CrossEntropyLoss
- **Scheduler:** CosineAnnealingLR — giảm learning rate theo hình cosine, giúp mô hình hội tụ mịn hơn.
- **Epochs:** 100
- **Early saving:** Lưu model có val accuracy cao nhất.

In [ ]:
# ── 4.1 Cấu hình huấn luyện ──
NUM_EPOCHS = 100
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

print(f"⚙️  Cấu hình huấn luyện:")
print(f"   Epochs       : {NUM_EPOCHS}")
print(f"   Batch size   : {BATCH_SIZE}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Weight decay : {WEIGHT_DECAY}")
print(f"   Scheduler    : CosineAnnealingLR (T_max={NUM_EPOCHS})")
print(f"   Loss fn      : CrossEntropyLoss")
print(f"   Optimizer    : Adam")

In [ ]:
# ── 4.2 Hàm huấn luyện 1 epoch ──
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        
        # Gradient clipping — ổn định huấn luyện Transformer
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        running_loss += loss.item() * X_batch.size(0)
        _, predicted = outputs.max(1)
        total += y_batch.size(0)
        correct += predicted.eq(y_batch).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


# ── 4.3 Hàm đánh giá ──
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            running_loss += loss.item() * X_batch.size(0)
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


print("✅ Đã định nghĩa hàm train_one_epoch() và evaluate().")

In [ ]:
# ── 4.4 Vòng lặp huấn luyện chính ──
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [],  'val_acc': [],
    'lr': [],         'epoch_time': []
}

best_val_acc = 0.0
best_epoch = 0
best_model_state = None

print(f"{'='*85}")
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>10} | {'Val Acc':>9} | {'LR':>10} | {'Time':>6}")
print(f"{'-'*85}")

total_train_time = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()
    
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    # Scheduler step
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    
    epoch_time = time.time() - epoch_start
    total_train_time += epoch_time
    
    # Ghi lịch sử
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    history['epoch_time'].append(epoch_time)
    
    # Lưu model tốt nhất
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        best_model_state = deepcopy(model.state_dict())
        torch.save(best_model_state, BEST_MODEL_PATH)
        marker = ' ◀ BEST'
    else:
        marker = ''
    
    # In tiến độ
    if epoch <= 10 or epoch % 10 == 0 or marker:
        print(
            f"{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.2%} | "
            f"{val_loss:>10.4f} | {val_acc:>8.2%} | {current_lr:>10.6f} | {epoch_time:>5.1f}s{marker}"
        )

print(f"{'-'*85}")
print(f"\n🏆 Kết quả tốt nhất:")
print(f"   Best Val Accuracy: {best_val_acc:.4%} (Epoch {best_epoch})")
print(f"   Tổng thời gian huấn luyện: {total_train_time:.1f}s")
print(f"   Thời gian TB/epoch: {total_train_time/NUM_EPOCHS:.2f}s")
print(f"   Model đã lưu: {BEST_MODEL_PATH}")

---
## 5. Trực quan hóa Quá trình Huấn luyện

In [ ]:
# ── 5.1 Training Curves ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, NUM_EPOCHS + 1)

# ── Loss ──
axes[0].plot(epochs_range, history['train_loss'], color=COLORS['train_loss'],
             label='Train Loss', linewidth=1.5, alpha=0.9)
axes[0].plot(epochs_range, history['val_loss'], color=COLORS['val_loss'],
             label='Val Loss', linewidth=1.5, alpha=0.9)
axes[0].axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.5, label=f'Best (Epoch {best_epoch})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('📉 Loss Curves')
axes[0].legend()

# ── Accuracy ──
axes[1].plot(epochs_range, [a * 100 for a in history['train_acc']],
             color=COLORS['train_acc'], label='Train Acc', linewidth=1.5, alpha=0.9)
axes[1].plot(epochs_range, [a * 100 for a in history['val_acc']],
             color=COLORS['val_acc'], label='Val Acc', linewidth=1.5, alpha=0.9)
axes[1].axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.5, label=f'Best (Epoch {best_epoch})')
axes[1].axhline(y=best_val_acc * 100, color=COLORS['accent'], linestyle=':',
                alpha=0.5, label=f'Best Val: {best_val_acc*100:.1f}%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('📈 Accuracy Curves')
axes[1].legend()

# ── Learning Rate ──
axes[2].plot(epochs_range, history['lr'], color=COLORS['transformer'],
             linewidth=2, alpha=0.9)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('📐 CosineAnnealingLR Schedule')
axes[2].ticklabel_format(axis='y', style='scientific', scilimits=(-4, -4))

plt.suptitle('Transformer Training Dashboard', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2 Epoch Time Distribution ──
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(epochs_range, history['epoch_time'], color=COLORS['transformer'], alpha=0.7, width=0.8)
ax.axhline(y=np.mean(history['epoch_time']), color=COLORS['secondary'],
           linestyle='--', linewidth=1.5, label=f"TB: {np.mean(history['epoch_time']):.2f}s")
ax.set_xlabel('Epoch')
ax.set_ylabel('Thời gian (s)')
ax.set_title('⏱️ Thời gian mỗi Epoch')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Đánh giá trên Tập Test

In [ ]:
# ── 6.1 Load best model & đánh giá ──
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average='weighted')

print(f"{'='*60}")
print(f"🎯 KẾT QUẢ TRÊN TẬP TEST")
print(f"{'='*60}")
print(f"   Test Accuracy : {test_acc:.4%}")
print(f"   Test F1-Score : {test_f1:.4f} (weighted)")
print(f"{'='*60}")

In [ ]:
# ── 6.2 Classification Report ──
print("\n📋 BÁO CÁO PHÂN LOẠI CHI TIẾT:")
print("=" * 60)
print(classification_report(
    all_labels, all_preds,
    target_names=le.classes_,
    digits=4
))

In [ ]:
# ── 6.3 Confusion Matrix ──
cm = confusion_matrix(all_labels, all_preds)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            ax=axes[0], linewidths=0.5)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix (Counts)')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='RdYlGn',
            xticklabels=le.classes_, yticklabels=le.classes_,
            ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix (Normalized %)')

plt.suptitle(f'Transformer — Test Accuracy: {test_acc:.2%}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. 🔍 Trực quan hóa Attention Weights

Đây là lợi thế lớn nhất của Transformer: ta có thể **nhìn trực tiếp** mô hình đang chú ý frame nào khi phân loại.

Ma trận Attention (15×15) cho biết:
- **Hàng $i$**: Frame $i$ đang "hỏi" (query)
- **Cột $j$**: Frame $j$ đang được "chú ý" (key)
- **Giá trị**: Mức độ quan trọng mà frame $i$ gán cho frame $j$

In [ ]:
# ── 7.1 Trích xuất Attention Maps ──
model.eval()

# Lấy 1 batch từ test set
sample_batch_x, sample_batch_y = next(iter(test_loader))
sample_batch_x = sample_batch_x.to(device)

# Lấy attention weights
_, attention_maps = model.get_attention_maps(sample_batch_x)

print(f"📊 Số Transformer layers: {len(attention_maps)}")
for i, attn in enumerate(attention_maps):
    print(f"   Layer {i}: attention shape = {attn.shape}")
    # Expected: (batch, nhead, 15, 15)

In [ ]:
# ── 7.2 Attention Heatmap cho 1 sample, tất cả heads ──
sample_idx = 0  # Sample đầu tiên trong batch
true_label = le.classes_[sample_batch_y[sample_idx].item()]

fig, axes = plt.subplots(len(attention_maps), 4, figsize=(16, 4 * len(attention_maps)))

frame_labels = [f'F{i}' for i in range(15)]

for layer_idx, attn in enumerate(attention_maps):
    attn_sample = attn[sample_idx].cpu().numpy()  # (nhead, 15, 15)
    
    for head_idx in range(4):
        ax = axes[layer_idx, head_idx] if len(attention_maps) > 1 else axes[head_idx]
        
        sns.heatmap(
            attn_sample[head_idx],
            ax=ax,
            cmap='magma',
            vmin=0, vmax=attn_sample[head_idx].max(),
            xticklabels=frame_labels,
            yticklabels=frame_labels,
            square=True,
            linewidths=0.3,
            cbar_kws={'shrink': 0.6}
        )
        ax.set_title(f'Layer {layer_idx+1}, Head {head_idx+1}', fontsize=10)
        ax.set_xlabel('Key (Frame)' if layer_idx == len(attention_maps) - 1 else '')
        ax.set_ylabel('Query (Frame)' if head_idx == 0 else '')

plt.suptitle(
    f'Self-Attention Maps — Sample #{sample_idx} (Label: "{true_label}")',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.3 Attention trung bình qua tất cả heads (Layer cuối) ──
# Layer cuối thường mang thông tin ngữ nghĩa cao nhất
last_layer_attn = attention_maps[-1][sample_idx].cpu().numpy()  # (nhead, 15, 15)
avg_attn = last_layer_attn.mean(axis=0)  # (15, 15) — trung bình qua 4 heads

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Heatmap
sns.heatmap(
    avg_attn,
    ax=axes[0],
    cmap='YlOrRd',
    annot=True, fmt='.3f',
    xticklabels=[f'Frame {i}' for i in range(15)],
    yticklabels=[f'Frame {i}' for i in range(15)],
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Attention Weight'}
)
axes[0].set_title(f'Avg Attention (Layer 3) — "{true_label}"', fontsize=12)
axes[0].set_xlabel('Key Frame (being attended to)')
axes[0].set_ylabel('Query Frame (attending)')

# Attention strength per frame (được chú ý bao nhiêu)
frame_importance = avg_attn.sum(axis=0)  # Tổng attention mà mỗi frame nhận
frame_importance = frame_importance / frame_importance.sum()  # Normalize

bars = axes[1].bar(
    range(15), frame_importance,
    color=plt.cm.YlOrRd(frame_importance / frame_importance.max()),
    edgecolor='gray', linewidth=0.5
)
axes[1].set_xlabel('Frame Index')
axes[1].set_ylabel('Normalized Attention Received')
axes[1].set_title('Frame Importance (tổng attention nhận được)', fontsize=12)
axes[1].set_xticks(range(15))
axes[1].set_xticklabels([f'F{i}' for i in range(15)])

# Đánh dấu frame quan trọng nhất
most_important = np.argmax(frame_importance)
axes[1].annotate(
    f'Most attended\nFrame {most_important}',
    xy=(most_important, frame_importance[most_important]),
    xytext=(most_important + 2, frame_importance[most_important] + 0.01),
    arrowprops=dict(arrowstyle='->', color=COLORS['secondary']),
    fontsize=9, color=COLORS['secondary'], fontweight='bold'
)

plt.tight_layout()
plt.show()

print(f"💡 Phân tích: Frame {most_important} nhận attention cao nhất ({frame_importance[most_important]:.3f}).")
print(f"   Điều này gợi ý rằng frame này chứa thông tin phân biệt quan trọng cho cử chỉ '{true_label}'.")

In [ ]:
# ── 7.4 So sánh Attention patterns giữa các nhãn khác nhau ──
# Lấy 1 sample từ mỗi lớp (nếu có trong batch)
unique_labels_in_batch = {}
for i in range(len(sample_batch_y)):
    lbl = sample_batch_y[i].item()
    if lbl not in unique_labels_in_batch:
        unique_labels_in_batch[lbl] = i

# Chọn tối đa 4 nhãn khác nhau để hiển thị
selected_labels = list(unique_labels_in_batch.items())[:4]
n_show = len(selected_labels)

fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 4.5))
if n_show == 1:
    axes = [axes]

for ax_idx, (lbl_id, samp_idx) in enumerate(selected_labels):
    attn_map = attention_maps[-1][samp_idx].cpu().numpy().mean(axis=0)  # (15, 15)
    label_name = le.classes_[lbl_id]
    
    sns.heatmap(
        attn_map, ax=axes[ax_idx],
        cmap='magma', square=True,
        xticklabels=[f'F{i}' for i in range(15)],
        yticklabels=[f'F{i}' for i in range(15)],
        linewidths=0.3,
        cbar=True,
        cbar_kws={'shrink': 0.6}
    )
    axes[ax_idx].set_title(f'"{label_name}"', fontsize=11, fontweight='bold')
    axes[ax_idx].set_xlabel('Key Frame')
    if ax_idx == 0:
        axes[ax_idx].set_ylabel('Query Frame')

plt.suptitle('Attention Patterns theo Loại Cử chỉ (Layer 3, Avg Heads)',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("💡 Nhận xét: Các cử chỉ khác nhau thường có attention pattern khác nhau —")
print("   cho thấy mô hình học được đặc trưng thời gian riêng biệt cho mỗi loại.")

---
## 8. 📊 So sánh LSTM vs Transformer

So sánh hai kiến trúc đã huấn luyện trong dự án.

In [ ]:
# ── 8.1 Thống kê Transformer ──
transformer_stats = {
    'Model': 'Transformer',
    'Parameters': f"{sum(p.numel() for p in model.parameters()):,}",
    'Avg Time/Epoch': f"{np.mean(history['epoch_time']):.2f}s",
    'Best Val Acc': f"{best_val_acc:.4%}",
    'Test Acc': f"{test_acc:.4%}",
    'Test F1': f"{test_f1:.4f}",
    'Best Epoch': best_epoch
}

# ── 8.2 Tải thông số LSTM (nếu có từ Tuần 6) ──
lstm_model_path = os.path.join(MODEL_DIR, 'best_lstm.pth')
lstm_available = os.path.exists(lstm_model_path)

if lstm_available:
    # Thử đọc thông tin LSTM từ checkpoint
    lstm_checkpoint = torch.load(lstm_model_path, map_location='cpu', weights_only=True)
    lstm_param_count = sum(p.numel() for p in lstm_checkpoint.values())
    lstm_stats = {
        'Model': 'LSTM (Tuần 6)',
        'Parameters': f"{lstm_param_count:,}",
        'Avg Time/Epoch': 'N/A (xem Tuần 6)',
        'Best Val Acc': 'N/A (xem Tuần 6)',
        'Test Acc': 'N/A (xem Tuần 6)',
        'Test F1': 'N/A (xem Tuần 6)',
        'Best Epoch': 'N/A'
    }
    print(f"✅ Đã tìm thấy model LSTM: {lstm_model_path}")
    print(f"   LSTM Parameters: {lstm_param_count:,}")
else:
    lstm_stats = {
        'Model': 'LSTM (Tuần 6)',
        'Parameters': '~80,000-150,000 (typical)',
        'Avg Time/Epoch': '~0.5-1.5s (typical)',
        'Best Val Acc': 'Chạy notebook Tuần 6 để biết',
        'Test Acc': 'Chạy notebook Tuần 6 để biết',
        'Test F1': 'N/A',
        'Best Epoch': 'N/A'
    }
    print(f"⚠️  Chưa tìm thấy model LSTM tại: {lstm_model_path}")
    print(f"   Dùng giá trị ước lượng cho bảng so sánh.")

# ── 8.3 Bảng so sánh ──
print(f"\n{'='*90}")
print(f"{'BẢNG SO SÁNH LSTM vs TRANSFORMER':^90}")
print(f"{'='*90}")
print(f"{'Chỉ số':<20} | {'LSTM (Tuần 6)':<30} | {'Transformer (Tuần 7)':<30}")
print(f"{'-'*90}")
print(f"{'Parameters':<20} | {lstm_stats['Parameters']:<30} | {transformer_stats['Parameters']:<30}")
print(f"{'Avg Time/Epoch':<20} | {lstm_stats['Avg Time/Epoch']:<30} | {transformer_stats['Avg Time/Epoch']:<30}")
print(f"{'Best Val Acc':<20} | {lstm_stats['Best Val Acc']:<30} | {transformer_stats['Best Val Acc']:<30}")
print(f"{'Test Accuracy':<20} | {lstm_stats['Test Acc']:<30} | {transformer_stats['Test Acc']:<30}")
print(f"{'Test F1 Score':<20} | {lstm_stats['Test F1']:<30} | {transformer_stats['Test F1']:<30}")
print(f"{'Best Epoch':<20} | {str(lstm_stats['Best Epoch']):<30} | {str(transformer_stats['Best Epoch']):<30}")
print(f"{'='*90}")

### 🔬 Phân tích chuyên sâu: LSTM vs Transformer cho chuỗi ngắn

#### Khi nào Transformer vượt trội?

| Yếu tố | Giải thích |
|---|---|
| **Chuỗi dài (>50 frames)** | Self-Attention "nhìn" mọi cặp frame — không bị quên frame đầu như LSTM |
| **Song song hóa** | Tất cả frames được xử lý đồng thời → nhanh hơn trên GPU |
| **Giải thích được** | Attention map trực quan: biết frame nào quan trọng |
| **Dữ liệu lớn** | Transformer scale tốt hơn khi có nhiều dữ liệu |

#### Khi nào LSTM chiếm ưu thế?

| Yếu tố | Giải thích |
|---|---|
| **Chuỗi rất ngắn (≤15 frames)** | Overhead của Self-Attention không cần thiết, LSTM xử lý tuần tự hiệu quả |
| **Dữ liệu ít** | Ít tham số hơn → ít overfitting hơn |
| **Tính thứ tự tự nhiên** | LSTM "tự biết" thứ tự — không cần Positional Encoding |
| **Bộ nhớ hạn chế** | Ít tham số hơn → inference nhanh trên edge device |

#### 📌 Kết luận cho bài toán Hand Gesture (15 frames)

Với chuỗi chỉ 15 frames:
- **LSTM** có lợi thế do tính tuần tự tự nhiên phù hợp với gesture data.
- **Transformer** mạnh hơn về khả năng giải thích (attention visualization) và có thể tốt hơn nếu tăng lượng dữ liệu.
- Nếu hai model cho accuracy tương đương, **LSTM nhẹ hơn** phù hợp deploy; **Transformer** phù hợp cho phân tích và nghiên cứu.

> 💡 **Thực tế:** Nhiều bài toán gesture recognition hiện đại kết hợp cả hai: dùng **CNN + LSTM** cho feature extraction tuần tự, và **Transformer** cho attention-based fusion.

---
## 9. Tổng kết & Kết luận

### 📊 Tóm tắt kết quả

In [ ]:
# ── 9.1 Summary card ──
print("╔" + "═" * 60 + "╗")
print("║" + " 🤖 TRANSFORMER — TÓM TẮT KẾT QUẢ TUẦN 7".ljust(60) + "║")
print("╠" + "═" * 60 + "╣")
print(f"║  Kiến trúc     : Transformer Encoder (3 layers, 4 heads)".ljust(61) + "║")
print(f"║  d_model       : 128, dim_ff: 256".ljust(61) + "║")
print(f"║  Input shape   : (batch, 15, 63)".ljust(61) + "║")
print(f"║  Tổng tham số  : {sum(p.numel() for p in model.parameters()):,}".ljust(61) + "║")
print("╠" + "═" * 60 + "╣")
print(f"║  Best Val Acc  : {best_val_acc:.4%} (Epoch {best_epoch})".ljust(61) + "║")
print(f"║  Test Accuracy : {test_acc:.4%}".ljust(61) + "║")
print(f"║  Test F1 Score : {test_f1:.4f}".ljust(61) + "║")
print(f"║  Avg Time/Epoch: {np.mean(history['epoch_time']):.2f}s".ljust(61) + "║")
print(f"║  Total Time    : {total_train_time:.1f}s".ljust(61) + "║")
print("╠" + "═" * 60 + "╣")
print(f"║  Optimizer     : Adam (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})".ljust(61) + "║")
print(f"║  Scheduler     : CosineAnnealingLR (T_max={NUM_EPOCHS})".ljust(61) + "║")
print(f"║  Model saved   : best_transformer.pth".ljust(61) + "║")
print("╚" + "═" * 60 + "╝")

### 📝 Những gì đã học được trong Tuần 7

1. **Self-Attention Mechanism**: Hiểu cách mỗi frame "truy vấn" thông tin từ tất cả các frame khác thông qua Q/K/V projection.

2. **Positional Encoding**: Khắc phục nhược điểm "không biết thứ tự" của Transformer bằng hàm sin/cos — tạo mã vị trí duy nhất cho mỗi frame.

3. **Multi-Head Attention**: 4 heads cho phép mô hình học 4 "góc nhìn" khác nhau về mối quan hệ giữa các frame — một head có thể tập trung vào frame liền kề, head khác nhìn frame đầu/cuối.

4. **Attention Visualization**: Trực quan hóa attention weights giúp **giải thích** tại sao mô hình đưa ra quyết định, frame nào được mô hình coi là quan trọng nhất.

5. **CosineAnnealingLR**: Scheduler giảm learning rate theo hình cosine — ban đầu giảm chậm, sau đó giảm nhanh — giúp mô hình tinh chỉnh mịn ở cuối quá trình huấn luyện.

---

### 🔜 Tiếp theo: Tuần 8 — Failure Analysis
- Phân tích chi tiết các trường hợp dự đoán sai.
- So sánh tổng thể tất cả kiến trúc: MLP → CNN → LSTM → Transformer.
- Đề xuất cải tiến và hướng phát triển.